# 📡 EDA-02 — Exploración de Telemetría y Disponibilidad de Sensores
## Capa Bronze · Turbina Kelmarsh T1 · 2018–2021

---

### Contexto

Una vez definido el catálogo de 41 fallos en `EDA-01`, necesitamos responder la segunda pregunta del proyecto:

> *¿Los sensores SCADA necesarios para predecir cada familia de fallos tienen suficientes datos disponibles?*

El dataset Bronze de telemetría contiene **303 columnas** y **210.388 filas** (una por cada intervalo de 10 minutos de 2018 a 2021).  
No todas las columnas tienen datos utilizables: algunas tienen hasta el 53% de valores nulos, lo que las hace inviables como features de entrenamiento.

### Objetivos de este notebook

1. **Limpiar los headers** de los CSV originales (contienen caracteres especiales incompatibles con Spark)
2. **Cargar y unir** los 4 años de telemetría en un único DataFrame
3. **Analizar la disponibilidad** de las 303 columnas (% de nulos por columna)
4. **Decidir qué columnas conservar** para cada familia de fallos

### Entradas y salidas

| | Ruta | Descripción |
|---|---|---|
| **Input** | `data/bronze/Kelmarsh_SCADA_YYYY_*/Turbine_Data_Kelmarsh_1_*.csv` | Telemetría SCADA T1, 4 archivos anuales |
| **Output** | Análisis de nulos documentado | Decisiones de descarte por columna |

---


## 1. Limpieza de Headers de los CSV Bronze

### El problema

Los archivos CSV de Kelmarsh provienen de la plataforma Greenbyte y tienen un formato no estándar:

- Las **primeras 9 líneas** son comentarios de metadatos que empiezan con `#`
- La **línea 10** es el header real, pero contiene **caracteres especiales** problemáticos:
  comas dentro de nombres de columna entre comillas, paréntesis, símbolos `°`, `/`, `+`
- Spark no puede inferir correctamente el esquema con este formato

### La solución

Implementamos un parser que:
1. Localiza la línea 10 (el header real)
2. Elimina las comillas y reemplaza las comas internas por guiones
3. Convierte todos los nombres de columna a `snake_case` limpio (solo alfanuméricos y `_`)
4. Genera una lista de rutas + headers limpios para cargar con Spark en el paso siguiente


In [1]:
import os
import glob
import re

base_dir = os.path.dirname(os.getcwd())
bronze_dir = os.path.join(base_dir, "data", "bronze")

target_years = ["2018", "2019", "2020", "2021"]

def clean_header_line(line):
    result = []
    in_quotes = False
    
    for char in line:
        if char == '"':
            in_quotes = not in_quotes
        elif char == ',' and in_quotes:
            result.append('-')
        else:
            result.append(char)
    
    cleaned = ''.join(result)
    if cleaned.startswith('# '):
        cleaned = cleaned[2:]
    elif cleaned.startswith('#'):
        cleaned = cleaned[1:]
    
    columns = cleaned.split(',')
    clean_columns = []
    for col in columns:
        col_clean = re.sub(r'[^a-zA-Z0-9_ ]', '', col)
        col_clean = col_clean.replace(' ', '_')
        col_clean = re.sub(r'_+', '_', col_clean)
        col_clean = col_clean.strip('_')
        clean_columns.append(col_clean)
    
    return ','.join(clean_columns)

def get_cleaned_files_info():
    """
    Devuelve lista de tuplas: (ruta_archivo, header_limpio)
    """
    all_files = []
    for year in target_years:
        pattern = os.path.join(bronze_dir, f"Kelmarsh_SCADA_{year}_*", f"Turbine_Data_Kelmarsh_1_*.csv")
        all_files.extend(sorted(glob.glob(pattern)))
    
    print(f"📁 Archivos encontrados: {len(all_files)}")
    
    files_info = []
    
    for file_path in all_files:
        filename = os.path.basename(file_path)
        print(f"\n🔄 Analizando: {filename}")
        
        # Leer solo las primeras 10 líneas para encontrar el header
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_number, line in enumerate(f, 1):
                if line_number == 10 and 'Date and time' in line:
                    cleaned_header = clean_header_line(line)
                    files_info.append({
                        'path': file_path,
                        'filename': filename,
                        'header': cleaned_header,
                        'skip_lines': 9  # saltar las 9 primeras (comentarios)
                    })
                    print(f"   ✅ Header limpio: {cleaned_header[:80]}...")
                    break
    
    print(f"\n✅ Total de archivos listos: {len(files_info)}")
    return files_info

# Variable global para el Script 2
files_info = get_cleaned_files_info()

📁 Archivos encontrados: 4

🔄 Analizando: Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
   ✅ Header limpio: Date_and_time,Wind_speed_ms,Wind_speed_Standard_deviation_ms,Wind_speed_Minimum_...

🔄 Analizando: Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   ✅ Header limpio: Date_and_time,Wind_speed_ms,Wind_speed_Standard_deviation_ms,Wind_speed_Minimum_...

🔄 Analizando: Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
   ✅ Header limpio: Date_and_time,Wind_speed_ms,Wind_speed_Standard_deviation_ms,Wind_speed_Minimum_...

🔄 Analizando: Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv
   ✅ Header limpio: Date_and_time,Wind_speed_ms,Wind_speed_Standard_deviation_ms,Wind_speed_Minimum_...

✅ Total de archivos listos: 4


---

## 2. Carga de Telemetría con Spark

Con los headers limpios, cargamos cada archivo anual como un DataFrame de Spark y los unimos en un único DataFrame consolidado.

**Decisión técnica — `unionByName` con `allowMissingColumns=True`:**  
Los archivos de distintos años pueden tener columnas ligeramente distintas (el firmware de la turbina se actualiza y añade o elimina señales). `unionByName` con esta opción rellena con `null` las columnas ausentes en años anteriores, preservando toda la información disponible sin pérdida.

**Resultado esperado:** ~210.000 filas × 303 columnas


In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from IPython.display import display, HTML
display(HTML("<style>div.output_area pre { max-height: none; }</style>"))

# ==============================================================================
# 1. SPARK SESSION
# ==============================================================================
spark = SparkSession.builder \
    .appName("Kelmarsh-EDA-Notebook") \
    .master("local[6]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# ==============================================================================
# 2. VERIFICAR QUE SCRIPT 1 SE EJECUTÓ
# ==============================================================================
try:
    files_info
except NameError:
    raise NameError("❌ Ejecuta primero el Script 1. La variable 'files_info' no existe.")

print(f"📁 Archivos a cargar: {len(files_info)}")
for info in files_info:
    print(f"   - {info['filename']}")

# ==============================================================================
# 3. CARGAR CADA ARCHIVO CON SPARK (LEYENDO COMO TEXTO, FILTRANDO COMENTARIOS)
# ==============================================================================
spark_dfs = []

for info in files_info:
    file_path = info['path']
    header = info['header']
    
    # Leer como RDD de texto para saltar las líneas de comentarios
    rdd = spark.sparkContext.textFile(file_path)
    
    # Filtrar: quitar líneas que empiecen con # (comentarios) y líneas vacías
    # La línea 10 (header) no empieza con # después de la limpieza del Script 1
    filtered_rdd = rdd.zipWithIndex().filter(lambda x: x[1] >= 9).map(lambda x: x[0])
    
    # Convertir RDD a DataFrame usando el header limpio
    # Primero, reconstruir el CSV con el header limpio como primera línea
    header_rdd = spark.sparkContext.parallelize([header])
    full_rdd = header_rdd.union(filtered_rdd)
    
    # Guardar temporalmente y leer con Spark CSV (más robusto para tipos)
    # O usar directamente from_csv si prefieres evitar disco
    
    # Método alternativo más limpio: leer con wholeTextFiles, procesar, y usar spark.read.csv
    # con una versión temporal del archivo
    
    print(f"\n🔹 Cargando: {info['filename']}")
    
    # OPCIÓN MÁS RÁPIDA: usar spark.read.csv con skipLines (Spark 3.3+)
    # o simplemente leer todo y filtrar después
    
    # Para Spark < 3.3, la forma más rápida es:
    raw_df = spark.read.text(file_path)
    
    # Filtrar líneas de comentarios (empiezan con #)
    data_df = raw_df.filter(~F.col("value").startswith("#"))
    
    # La primera fila ahora es el header original (línea 10 del archivo)
    # Necesitamos parsear el CSV manualmente o usar un truco
    
    # MÉTODO MÁS EFICIENTE: escribir un archivo temporal limpio y leerlo
    temp_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "temp_spark")
    os.makedirs(temp_dir, exist_ok=True)
    
    # Leer archivo original, saltar 9 líneas, poner header limpio
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()[9:]  # saltar las 9 primeras
    
    # Insertar header limpio
    lines.insert(0, header + '\n')
    
    # Escribir temporal
    temp_file = os.path.join(temp_dir, f"temp_{info['filename']}")
    with open(temp_file, 'w', encoding='utf-8') as f:
        f.writelines(lines)
    
    # Leer con Spark (ahora es un CSV limpio estándar)
    sdf = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(temp_file)
    
    spark_dfs.append(sdf)
    print(f"   ✅ {sdf.count()} filas x {len(sdf.columns)} columnas")

# ==============================================================================
# 4. UNIR TODOS LOS DATAFRAMES
# ==============================================================================
if len(spark_dfs) == 1:
    turbine_data_df = spark_dfs[0]
else:
    turbine_data_df = spark_dfs[0]
    for sdf in spark_dfs[1:]:
        turbine_data_df = turbine_data_df.unionByName(sdf, allowMissingColumns=True)

# ==============================================================================
# 5. RESULTADOS
# ==============================================================================
print("\n" + "="*60)
print("VISTA PREVIA DE LAS PRIMERAS 5 FILAS:")
turbine_data_df.show(5, truncate=False)

total_records = turbine_data_df.count()
print(f"\n🎯 Total de registros: {total_records}")
print(f"📊 NÚMERO TOTAL DE COLUMNAS: {len(turbine_data_df.columns)}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/31 21:30:26 WARN Utils: Your hostname, medion, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlo1)
26/05/31 21:30:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 21:30:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


📁 Archivos a cargar: 4
   - Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
   - Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   - Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
   - Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv



🔹 Cargando: Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv


   ✅ 52561 filas x 299 columnas

🔹 Cargando: Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv


   ✅ 52561 filas x 299 columnas

🔹 Cargando: Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv


   ✅ 52705 filas x 299 columnas

🔹 Cargando: Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv


   ✅ 52561 filas x 303 columnas

VISTA PREVIA DE LAS PRIMERAS 5 FILAS:


26/05/31 21:30:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------+------------------+------------------------------------+-------------------------+-------------------------+--------------------+-------------------------+---------------------------------------------+----------------------------------+----------------------------------+-------------------------+---------------------------------------------+----------------------------------+----------------------------------+---------------------------------+------------------+--------------------+--------------------------------------+---------------------------+---------------------------+----------------------------------------+-----------------------------+-----------------------------+---------------------+--------------------------+--------------------------+-----------------------------+-------------------+---------------------------+-------------------+---------------------------+-----------------------------------+--------------------------------------------+-------------

---

## 3. Análisis de Disponibilidad de Sensores (Nulos por Columna)

### Por qué este análisis es crítico para el proyecto

En ML supervisado, un sensor con el 50% de sus valores como `NaN` no es solo "datos que faltan" —  
es un sensor que estuvo inactivo la mitad del tiempo. Usar esas columnas como features tiene dos efectos negativos:

1. **El modelo aprende el patrón de ausencia**, no el patrón físico del sensor
2. **Las ventanas rodantes** (media, std, pendiente de 24h/7d) se calculan mal: con la mitad de los puntos, la media no representa la realidad operativa

**Umbral de descarte adoptado: > 40% de nulos**

Por encima del 40%, la columna tiene más huecos que datos y no es fiable como feature primaria.  
Las columnas entre 10–40% se conservan pero se documenta que necesitan imputación.  
Las columnas con < 10% de nulos son directamente utilizables.


In [3]:
from pyspark.sql import functions as F


# ==============================================================================
# ANÁLISIS DE NULOS Y NaN POR COLUMNA
# ==============================================================================
print("=" * 80)
print("📊 ANÁLISIS DE VALORES NULOS Y NaN POR COLUMNA")
print("=" * 80)

# Crear expresiones de agregación para contar nulos + NaN de todas las columnas a la vez
agg_exprs = []
for i, col_name in enumerate(turbine_data_df.columns):
    # Detectar: isNull() OR (casteado a string == 'NaN')
    agg_exprs.append(
        F.sum(
            F.when(
                F.col(f"`{col_name}`").isNull() | (F.col(f"`{col_name}`").cast("string") == 'NaN'),
                1
            ).otherwise(0)
        ).alias(f"null_col_{i}")
    )

# Ejecutar una sola agregación
null_summary = turbine_data_df.agg(*agg_exprs).collect()[0]
total_rows = turbine_data_df.count()

# Procesar resultados
null_counts = []
for i, col_name in enumerate(turbine_data_df.columns):
    null_count = null_summary[f"null_col_{i}"]
    null_pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
    null_counts.append((col_name, null_count, null_pct))

# Ordenar por nulos descendente
null_counts_sorted = sorted(null_counts, key=lambda x: x[1], reverse=True)

print(f"\n📋 Total de filas: {total_rows}")
print(f"📋 Total de columnas: {len(turbine_data_df.columns)}")

cols_with_nulls = [(c, n, p) for c, n, p in null_counts_sorted if n > 0]
print(f"\n🔴 Columnas con nulos/NaN: {len(cols_with_nulls)} de {len(turbine_data_df.columns)}")

if cols_with_nulls:
    print("\n" + "-" * 80)
    print(f"{'Columna':<50} {'Nulos+NaN':>10} {'% Total':>10}")
    print("-" * 80)
    for col_name, null_count, null_pct in cols_with_nulls:
        print(f"{col_name:<50} {null_count:>10} {null_pct:>9.2f}%")
    
else:
    print("\n✅ No hay columnas con valores nulos ni NaN")

# Columnas completamente vacías (100% nulos/NaN)
fully_null = [(c, n) for c, n, p in null_counts_sorted if p == 100]
if fully_null:
    print(f"\n⚠️  COLUMNAS COMPLETAMENTE VACÍAS (100% nulos/NaN): {len(fully_null)}")
    for col_name, _ in fully_null:
        print(f"   - {col_name}")

# Columnas prácticamente vacías (≥99% nulos/NaN) — útil para decidir si descartar
mostly_null = [(c, n, p) for c, n, p in null_counts_sorted if p >= 99.0 and n < total_rows]
if mostly_null:
    print(f"\n🟡 COLUMNAS PRÁCTICAMENTE VACÍAS (≥99% nulos/NaN): {len(mostly_null)}")
    for col_name, null_count, null_pct in mostly_null:
        print(f"   - {col_name}: {null_count}/{total_rows} ({null_pct:.2f}%)")

# Columnas sin nulos ni NaN
cols_no_nulls = [c for c, n, p in null_counts if n == 0]
print(f"\n🟢 Columnas sin nulos ni NaN: {len(cols_no_nulls)}")
if cols_no_nulls:
    print(f"   {', '.join(cols_no_nulls[:10])}{'...' if len(cols_no_nulls) > 10 else ''}")

📊 ANÁLISIS DE VALORES NULOS Y NaN POR COLUMNA


26/05/31 21:30:52 WARN DAGScheduler: Broadcasting large task binary with size 1153.2 KiB



📋 Total de filas: 210388
📋 Total de columnas: 303

🔴 Columnas con nulos/NaN: 295 de 303

--------------------------------------------------------------------------------
Columna                                             Nulos+NaN    % Total
--------------------------------------------------------------------------------
Manufacturer_Potential_Power_SCADA_kW                  210387    100.00%
Potential_power_met_mast_anemometer_kW                 210384    100.00%
Potential_power_met_mast_anemometer_MPC_kW             210384    100.00%
Equivalent_Full_Load_Hours_counter_s                   210384    100.00%
Potential_power_estimated_kW                           207517     98.64%
Reactive_Energy_Export_counter_kvarh                   195666     93.00%
Energy_Import_counter_kWh                              187635     89.19%
Operating_Performance_Ratio                            167017     79.39%
Lost_Production_Contractual_Custom_kWh                 158334     75.26%
Timebased_Contract

---

## 📋 Conclusiones del Notebook EDA-02

### Hallazgos del análisis de disponibilidad

| Categoría | Columnas | Criterio |
|-----------|----------|---------|
| ✅ **Utilizables directamente** | ~47 | < 10% nulos |
| ⚠️ **Utilizables con imputación** | ~3 | 10–40% nulos |
| ❌ **Descartadas** | ~249 | > 40% nulos o irrelevantes |

### Columnas descartadas y justificación por grupo

---

#### ❌ Grupo A — Columnas prácticamente vacías (≥ 99% nulos)
Nunca tuvieron datos reales en el período 2018–2021. Probablemente sensores no instalados en esta turbina o señales del firmware que no se activaron.

```
Manufacturer_Potential_Power_SCADA_kW    100%
Potential_power_met_mast_anemometer_kW   100%
Potential_power_met_mast_anemometer_MPC  100%
Equivalent_Full_Load_Hours_counter_s     100%
```

---

#### ❌ Grupo B — Columnas Max/Min/StdDev con ~53% nulos
Las columnas de estadísticos intra-intervalo (máximo, mínimo, desviación típica dentro de cada ventana de 10 min) solo están disponibles en aproximadamente la mitad del período. Esto indica un cambio de versión del firmware SCADA que comenzó a registrarlas después de 2019–2020.

**Impacto en el proyecto:** En lugar de usar estas columnas directamente, calcularemos los estadísticos equivalentes mediante **ventanas rodantes** sobre el valor medio en la Fase 3 del pipeline. Así obtenemos la misma información con el 98% de los datos disponibles.

Columnas afectadas (muestra):
```
Cable_windings_from_calibration_point_Max/Min/StdDev    53.41%
Tower_Acceleration_X/Y_Max/Min/StdDev                  53.41%
Drive_train_acceleration_Max/Min/StdDev                 52.31%
Motor_current_axis_1/2/3_Max/Min/StdDev                 52.30%
Gear_oil_inlet_pressure_Max/Min                         52.29%
Ambient_temperature_converter_Max/StdDev                52.29%
```

---

#### ❌ Grupo C — Métricas de producción y disponibilidad (0–5% nulos pero irrelevantes)
Columnas como `Lost_Production_*`, `Energy_Export_counter`, `Timebased_IEC_*`, `Production_Factor` y similares no son señales físicas de la turbina sino **indicadores de negocio derivados**. No tienen relación causal con el estado mecánico o eléctrico de la turbina y añadirían ruido al modelo.

---

#### ❌ Grupo D — Columnas de voltaje con ~39% nulos
```
Voltage_L1_U_Min/Max/StdDev   39.33%
Voltage_L2_V_Min/Max/StdDev   39.33%
Voltage_L3_W_Min/Max/StdDev   39.33%
```
Están en el límite del umbral (< 40%) pero los valores medios de las fases `Voltage_L1_U_V`, `L2`, `L3` tienen solo 3.71% de nulos y son suficientes para detectar desequilibrio de fases.

---

### Columnas conservadas para el entrenamiento (< 10% nulos)

Las columnas con menos del 10% de nulos cubren los **7 grupos de sensores** necesarios para las 4 familias de fallos activas:

| Grupo | Sensores clave | Nulos |
|-------|---------------|-------|
| Viento y orientación | `Wind_speed_ms`, `Wind_direction`, `Nacelle_position`, `Vane_position_12` | < 4% |
| Potencia y electricidad | `Power_kW`, `Power_factor_cosphi`, `Reactive_power_kvar`, `Grid_voltage_V` | < 4% |
| Tren de transmisión | `Generator_RPM_RPM`, `Drive_train_acceleration_mmss`, `Rotor_speed_RPM` | < 2% |
| Temperaturas generador | `Generator_bearing_front/rear_temperature_C`, `Stator_temperature_1_C` | < 8% |
| Temperaturas nacelle | `Nacelle_temperature_C`, `Nacelle_ambient_temperature_C`, `Ambient_temperature_converter_C` | < 4% |
| Hidráulico y aceite | `Gear_oil_inlet_pressure_bar`, `Gear_oil_pump_pressure_bar`, `Gear_oil_temperature_C` | < 4% |
| Pitch y cable | `Blade_angle_A/B/C`, `Motor_current_axis_1/2/3_A`, `Cable_windings_from_calibration_point` | < 4% |
| Desgaste mecánico | `Metal_particle_count` | **0%** |

---

### Decisiones sobre familias de fallos basadas en disponibilidad de sensores

| Familia | Sensores clave disponibles | Veredicto |
|---------|--------------------------|-----------|
| **Yaw/Cable** | `Cable_windings` (3.71%), `Nacelle_position` (1.36%), `Vane_position_12` (3.71%) | ✅ Todos disponibles |
| **Generador/Fans** | `Generator_bearing_*` (<8%), `Ambient_temp_converter` (1.36%) | ✅ Todos disponibles |
| **Freno/Hidráulico** | `Gear_oil_inlet_pressure` (1.36%), `Metal_particle_count` (0%) | ✅ Todos disponibles |
| **Pitch/Baterías** | `Motor_current_axis_*` (1.36%), `Blade_angle_*` (3.71%) | ✅ Disponibles (Max/StdDev calculados por rolling) |
| **Sensores anem/vane** | `Wind_speed_Sensor_1/2` disponibles | ❌ Descartado por concentración temporal (ver EDA-01) |
| **Torre/Vibración** | `Tower_Acceleration_X/Y` disponibles pero resolución insuficiente | ❌ Descartado por incompatibilidad física |

---

### Siguiente paso

**Notebook `03_merge_y_limpieza.ipynb`** — carga de telemetría filtrada a 50 columnas, guardado en Parquet y join con el `fault_log.csv` de Silver.
